# CPU notebook 3: baselines, headroom and realism (paper, RQ1, RQ4, RQ6, RQ7)

Tree policy, invocation rate only, state-aware trees and the perfect predictor; then the tree policy with a 2 s control
delay, loading time charged as memory and random cold-start durations. Azure 2021 and Huawei day 30, seeds 0-2.

**Setup.** Attach the code dataset and the data dataset; accelerator None (CPU). *Save Version → Save & Run All*.

In [ ]:
# 1. Find the code and data (any depth under /kaggle/input). No pip install: the code is put on Python's path.
import glob, os, shutil, sys
print('looking for the code ...', flush=True)
hits = [p for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), 'src', 'stawarm'))]
print('found:', hits, flush=True)
assert hits, 'Code not found. Check that the stawarm-code dataset is attached (Add Input) and contains pyproject.toml and src/.'
# Use the copy of this repository (it has reproduce.py); older copies in the same dataset are ignored.
current = [p for p in hits if os.path.exists(os.path.join(os.path.dirname(p), 'reproduce.py'))]
assert current, 'The attached stawarm-code dataset holds an older version of the code; upload the current repository as a new version.'
code_dir = os.path.dirname(current[0])
WORK = '/kaggle/working/stawarm'
if not os.path.exists(WORK):
    shutil.copytree(code_dir, WORK)
os.chdir(WORK)
# The repository ships the paper's result files in results/; keep them aside for comparison so that this
# notebook writes only its own outputs to results/.
if os.path.isdir('results') and not os.path.isdir('results_paper'):
    shutil.move('results', 'results_paper')
os.makedirs('results', exist_ok=True)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')          # used by every !python command below
sys.path.insert(0, os.path.join(WORK, 'src'))
os.makedirs('data/processed', exist_ok=True)
for name in ['azure2021_events.parquet', 'huawei_r2_day30_events.parquet', 'huawei_r2_cold_start_stats.parquet']:
    src = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    print(name, '->', src, flush=True)
    assert src, f'{name} not found: check that the stawarm-data dataset is attached'
    if not os.path.exists(f'data/processed/{name}'):
        os.symlink(src[0], f'data/processed/{name}')
import stawarm
print('code from', code_dir, '| package OK:', stawarm.__file__)
print('data:', os.listdir('data/processed'), flush=True)

In [ ]:
# 2. Record the software and hardware environment of this run
!python scripts/tools/record_environment.py results/environment_03_cpu_baselines_and_robustness.txt

In [ ]:
# 3. Tests (should end with '35 passed')
!python -m pytest -q 2>&1 | tail -2

## 4. Azure 2021: policies + oracle (about 65 min per seed)

In [ ]:
for seed in [0, 1, 2]:
    !python -u scripts/experiments/run_policies.py --variants app_nolead,rate,state_aware --risk-on none --lead-feature --oracle --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --seed {seed} --tag baselines_s{seed} 2>&1 | grep -v Warning

## 5. Azure 2021: realism (about 20 min per seed)

In [ ]:
for seed in [0, 1, 2]:
    !python -u scripts/experiments/run_policies.py --variants app_nolead --risk-on none --lead-feature --control-delay 2 --charge-loading --random-durations --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --seed {seed} --tag conservative_s{seed} 2>&1 | grep -v Warning

## 6. Huawei day 30: policies + oracle, and realism (about 25 min per seed)

In [ ]:
for seed in [0, 1, 2]:
    !python -u scripts/experiments/run_policies.py --dataset huawei_day30 --variants app_nolead,rate,state_aware --risk-on none --lead-feature --oracle --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --seed {seed} --tag baselines_s{seed} 2>&1 | grep -v Warning
    !python -u scripts/experiments/run_policies.py --dataset huawei_day30 --variants app_nolead --risk-on none --lead-feature --control-delay 2 --charge-loading --random-durations --thetas 0.02,0.05,0.1,0.15,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.8,0.9 --seed {seed} --tag conservative_s{seed} 2>&1 | grep -v Warning

## 7. Summaries at matched memory

In [ ]:
!python scripts/analysis/summarize_runs.py "results/policies_azure2021_baselines_s*" --out results/summary_baselines_azure.txt
!python scripts/analysis/summarize_runs.py "results/policies_azure2021_d2_load_rand_conservative_s*" --out results/summary_conservative_azure.txt
!python scripts/analysis/summarize_runs.py "results/policies_huawei_day30_baselines_s*" --out results/summary_baselines_huawei.txt
!python scripts/analysis/summarize_runs.py "results/policies_huawei_day30_d2_load_rand_conservative_s*" --out results/summary_conservative_huawei.txt

In [ ]:
# 8. Compare this run with the paper's result files (zero differences expected for tree-based runs)
!python scripts/analysis/compare_results.py results results_paper --out results/comparison_03_cpu_baselines_and_robustness.txt

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results_cpu_baselines', 'zip', 'results')
print('packed results -> results_cpu_baselines.zip')